In [1]:
!pip install requests python-dotenv --quiet

In [2]:
pip install pandas requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
import requests
import pandas as pd
import time
import os

from dotenv import load_dotenv

load_dotenv()  # membaca isi file .env

api_key = os.getenv("api_key")

if api_key:
    print("API key berhasil dimuat.")
else:
    print("API key belum ketemu. Pastikan file .env sudah dibuat dan diisi dengan benar.")

API key berhasil dimuat.


**BAGIAN 2 : MENCOBA MEMANGGIL API**

In [5]:
alamat_api = "https://api.stlouisfed.org/fred/series/observations"

parameter = {
    "series_id" : "DGS10",
    "api_key"   : api_key,
    "file_type" : "json"
}

response = requests.get(alamat_api, params=parameter)
print(f"Status Code: {response.status_code}")

hasil = response.json()
print(f"Jumlah data ditemukan (count)        : {hasil['count']}")
print(f"Jumlah observasi yang dikirim kali ini: {len(hasil['observations'])}")

Status Code: 200
Jumlah data ditemukan (count)        : 16885
Jumlah observasi yang dikirim kali ini: 16885


In [6]:
# Lihat bentuk data satu observasi

observasi_pertama = hasil["observations"][0]
observasi_pertama



{'realtime_start': '2026-09-22',
 'realtime_end': '2026-09-22',
 'date': '1962-01-02',
 'value': '4.06'}

In [7]:
# Lihat tipe datanya dari observasi_pertama
print(type(observasi_pertama["date"]))
print(type(observasi_pertama["value"]))

<class 'str'>
<class 'str'>


**MEMBUNGKUS JADI CLASS**

In [8]:
class Fred:

    def __init__(self, api_key):
        self.api_key = api_key
        self.alamat_api = "https://api.stlouisfed.org/fred/series/observations"

    def ambil_data(self, series_id, observation_date=None):
        parameter = {
            "series_id"  : series_id,
            "api_key"    : self.api_key,
            "file_type"  : "json"
        }
        if observation_date:
            parameter["observation_start"] = observation_date

        try:
            response = requests.get(self.alamat_api, params=parameter, timeout=20)
        except Exception:
            print("Koneksi bermasalah, mencoba lagi...")
            time.sleep(3)
            response = requests.get(self.alamat_api, params=parameter, timeout=20)

        if response.status_code != 200:
            print(f"Gagal mengambil data. Status: {response.status_code}")
            return pd.DataFrame()

        daftar_observasi = response.json()["observations"]

        data = []
        for observasi in daftar_observasi:
            data.append({
                "Tanggal" : observasi["date"],
                "Nilai"   : observasi["value"]
            })

        return pd.DataFrame(data)


print("Class Fred siap dipakai!")

Class Fred siap dipakai!


In [9]:
# Memanggil Class

client = Fred(api_key=api_key)
df_raw = client.ambil_data("DGS10", "2011-01-01")

print(f"Jumlah baris yang didapat: {len(df_raw)}")
df_raw.head()

Jumlah baris yang didapat: 4101


,Tanggal,Nilai
0,2011-01-03,3.36
1,2011-01-04,3.36
2,2011-01-05,3.50
3,2011-01-06,3.44
4,2011-01-07,3.34


**BAGIAN 4 : MEMBERSIHKAN DATA**

In [10]:
# Lihat tipe data 
print("1. Jumlah sel kosong per kolom:")
print(df_raw.isnull().sum())
print()

print("2. Jumlah baris yang kembar (berdasarkan Tanggal):")
print(df_raw.duplicated(subset="Tanggal").sum())
print()

print("3. Tipe data setiap kolom:")
print(df_raw.dtypes)

1. Jumlah sel kosong per kolom:
Tanggal    0
Nilai      0
dtype: int64

2. Jumlah baris yang kembar (berdasarkan Tanggal):
0

3. Tipe data setiap kolom:
Tanggal    str
Nilai      str
dtype: object


In [ ]:
# dtype kemungkinan adalah kolom tanggal yang maish bertipe object


In [16]:
#Menangani Nilai Kosong


def bersihkan_nilai(teks):
    if pd.isna(teks):
        return None
    if teks == ".":          # FRED pakai "." untuk data kosong
        return None
    return teks


df_raw["Nilai"] = df_raw["Nilai"].apply(bersihkan_nilai)

jumlah_sebelum = len(df_raw)
df_raw = df_raw.dropna(subset=["Nilai"])
print(f"Baris tanpa Nilai yang dibuang: {jumlah_sebelum - len(df_raw)}")

print()
print("Sel kosong setelah ditangani:")
print(df_raw.isnull().sum())

Baris tanpa Nilai yang dibuang: 0

Sel kosong setelah ditangani:
Tanggal    0
Nilai      0
dtype: int64


In [18]:
# Menangani tipe data

def ubah_ke_tanggal(teks):
    return pd.to_datetime(teks)

def ubah_ke_angka(teks):
    return float(teks)


print("Tipe data Tanggal sebelum:", df_raw["Tanggal"].dtype)
df_raw["Tanggal"] = df_raw["Tanggal"].apply(ubah_ke_tanggal)
print("Tipe data Tanggal sesudah:", df_raw["Tanggal"].dtype)

print("Tipe data Nilai sebelum:", df_raw["Nilai"].dtype)
df_raw["Nilai"] = df_raw["Nilai"].apply(ubah_ke_angka)
print("Tipe data Nilai sesudah:", df_raw["Nilai"].dtype)

df_raw.head()

Tipe data Tanggal sebelum: datetime64[us]
Tipe data Tanggal sesudah: datetime64[us]
Tipe data Nilai sebelum: str
Tipe data Nilai sesudah: float64


,Tanggal,Nilai
0,2011-01-03,3.36
1,2011-01-04,3.36
2,2011-01-05,3.50
3,2011-01-06,3.44
4,2011-01-07,3.34


In [20]:
# Menangani baris kembar

jumlah_sebelum = len(df_raw)

df_bersih = df_raw.drop_duplicates(subset="Tanggal")

print(f"Jumlah baris sebelum : {jumlah_sebelum}")
print(f"Jumlah baris sesudah : {len(df_bersih)}")
print(f"Baris kembar dibuang : {jumlah_sebelum - len(df_bersih)}")

Jumlah baris sebelum : 3931
Jumlah baris sesudah : 3931
Baris kembar dibuang : 0


In [21]:
# Pemeriksaan terakhir sebelum selesai

print(f"Jumlah baris           : {len(df_bersih)}")
print(f"Sudah lebih dari 100?  : {len(df_bersih) >= 100}")
print(f"Nilai masih ada kosong : {df_bersih['Nilai'].isnull().sum()}")
print(f"Tanggal masih kembar   : {df_bersih['Tanggal'].duplicated().sum()}")
print(f"Tipe kolom Tanggal     : {df_bersih['Tanggal'].dtype}")
print(f"Tipe kolom Nilai       : {df_bersih['Nilai'].dtype}")

Jumlah baris           : 3931
Sudah lebih dari 100?  : True
Nilai masih ada kosong : 0
Tanggal masih kembar   : 0
Tipe kolom Tanggal     : datetime64[us]
Tipe kolom Nilai       : float64


**BAGIAN 5 : MENYIMPAN HASIL**

In [22]:
df_bersih.to_csv("dataset_suku_bunga.csv", index=False)
print("Data berhasil disimpan ke file: dataset_suku_bunga.csv")

df_cek = pd.read_csv("dataset_suku_bunga.csv")
print(f"File terbaca kembali: {len(df_cek)} baris, {len(df_cek.columns)} kolom")
df_cek.head()

Data berhasil disimpan ke file: dataset_suku_bunga.csv
File terbaca kembali: 3931 baris, 2 kolom


,Tanggal,Nilai
0,2011-01-03,3.36
1,2011-01-04,3.36
2,2011-01-05,3.50
3,2011-01-06,3.44
4,2011-01-07,3.34
